# Regression NBA Model


## Configuration

## Imports

In [3]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from nba_ou.data_preparation.missing_data.clean_df_for_training import (
    clean_dataframe_for_training,
)
from nba_ou.modeling.modeling import (
    TemporalDecaySampleWeightRegressor,
    evaluate_day_by_day_walk_forward,
    split_latest_dates_holdout,
    make_walk_forward_last_n_seasons_splits,
    validate_time_splits,
    make_test_anchored_walk_forward_splits,
    assert_valid_time_splits,
    save_model_bundle,
    load_model_bundle,
)


In [4]:
TARGET_COL = "LINE_ERROR"
SAMPLE_WEIGHT_LAMBDA = 0.0075
SAMPLE_WEIGHT_LAMBDA_BOUNDS = (1e-4, 0.015)
TRAIN_GAMES = 2500

## Load Data

In [5]:
nan_threshold = 50.0
max_na_per_row = 80


data_path = "/home/adrian_alvarez/Projects/NBA_over_under_predictor/data/train_data/"
name = "all_odds_training_data_until_20260405.csv"

path = data_path + name

header_cols = pd.read_csv(path, nrows=0).columns
dtype_dict = {col: str for col in header_cols if "ID" in col.upper()}

df_stats = pd.read_csv(
    path,
    dtype=dtype_dict,
)
df_stats["GAME_DATE"] = pd.to_datetime(df_stats["GAME_DATE"]).dt.strftime("%Y-%m-%d")
df_stats = df_stats[df_stats['SEASON_YEAR'] >= 2021].copy()


In [6]:
exclude = "fanatics_sportsbook"

In [7]:
df_to_train = clean_dataframe_for_training(df_stats, nan_threshold=nan_threshold, max_na_per_row=max_na_per_row, create_missing_flags=False, verbose=1, keep_columns=['GAME_DATE'], exclude_cols_containing=[exclude])

STARTING DATAFRAME CLEANING PIPELINE
Starting basic cleaning with 6445 rows
Basic cleaning complete: 6430 rows remaining

Starting advanced column cleaning with 2948 columns

Advanced column cleaning complete: 2948 → 2054 columns (894 removed)


Applying missing data policy...

Missing Data Policy Report:
  Rows dropped: 0 (0.0%)
  Critical columns requiring data: 4
  Columns zero-filled: 112
  Infer pairs applied: 0/106
  Remaining NaN cells: 251290

Dropping rows with more than 80 NaN values...
Removed 618 rows exceeding NaN threshold
CLEANING COMPLETE
Final shape: (5812, 2054)


In [8]:
# Count NAs per column
na_counts = df_to_train.isna().sum()

# Get most common SEASON_YEAR for nulls in each column
most_common_season = []
for col in df_to_train.columns:
    if na_counts[col] > 0:
        null_rows = df_to_train[df_to_train[col].isna()]
        if len(null_rows) > 0 and "SEASON_YEAR" in df_to_train.columns:
            common_season = null_rows["SEASON_YEAR"].mode()
            most_common_season.append(
                common_season.iloc[0] if len(common_season) > 0 else None
            )
        else:
            most_common_season.append(None)
    else:
        most_common_season.append(None)

na_counts_df = pd.DataFrame(
    {
        "Column": na_counts.index,
        "NA_Count": na_counts.values,
        "NA_Percentage": (na_counts.values / len(df_to_train) * 100).round(2),
        "Most_Common_Season_Year": most_common_season,
    }
).sort_values("NA_Count", ascending=False)

na_counts_df[na_counts_df["NA_Count"] > 0]

,Column,NA_Count,NA_Percentage,Most_Common_Season_Year
1732,total_consensus_pct_under_TREND_SLOPE_LAST_5_H...,792,13.63,2023.0
1730,total_consensus_pct_over_TREND_SLOPE_LAST_5_HO...,778,13.39,2023.0
1736,spread_consensus_pct_home_TREND_SLOPE_LAST_5_H...,709,12.20,2023.0
1734,spread_consensus_pct_away_TREND_SLOPE_LAST_5_H...,702,12.08,2023.0
1731,total_consensus_pct_under_TREND_SLOPE_LAST_5_G...,684,11.77,2023.0
...,...,...,...,...
1885,spread_betmgm_price_away,1,0.02,2021.0
1886,spread_betmgm_price_home,1,0.02,2021.0
1875,total_betmgm_price_under,1,0.02,2021.0
1874,total_betmgm_price_over,1,0.02,2021.0


In [9]:
BET365_LINE_COL = "TOTAL_LINE_bet365"
# BET365_LINE_COL = "total_bet365_line_over"

# Ensure the main scoring line and actual total exist.
df_to_train = df_to_train.dropna(subset=[BET365_LINE_COL, "TOTAL_POINTS"]).copy()

In [10]:
df_to_train["LINE_ERROR"] = df_to_train["TOTAL_POINTS"] - df_to_train[BET365_LINE_COL]


In [11]:
df_to_train["GAME_DATE"] = pd.to_datetime(df_to_train["GAME_DATE"])
df_to_train = df_to_train.sort_values("GAME_DATE").reset_index(drop=True)

# Count games per season
games_per_season = df_to_train.groupby("SEASON_YEAR").size()
print(games_per_season)


SEASON_YEAR
2021    1239
2022    1235
2023    1010
2024    1238
2025    1090
dtype: int64


## Train / Test

In [12]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    root_mean_squared_error,
)
from sklearn.model_selection import cross_validate
from xgboost import XGBRegressor

from nba_ou.modeling.optuna_error_line import (
    fit_best_xgb_error_line,
    select_best_trial_lexicographic,
    summarize_lexicographic_candidates,
    summarize_optuna_trials,
    tune_xgb_error_line_optuna,
)
from nba_ou.modeling.scorers import (
    OverUnderScorerLineError,
    OverUnderScorerLineErrorMinEdge,
    evaluate_error_thresholds,
    over_under_betting_accuracy_error_line,
    over_under_betting_accuracy_error_line_with_min_edge,
)

In [13]:
df_dev, df_test_final = split_latest_dates_holdout(
    df=df_to_train,
    date_col="GAME_DATE",
    test_size=0.08,
)

print(f"Development set size: {len(df_dev)}")
print(f"Final test set size: {len(df_test_final)}")
print(
    "Final test date range:",
    df_test_final["GAME_DATE"].min(),
    "->",
    df_test_final["GAME_DATE"].max(),
)

Development set size: 5341
Final test set size: 471
Final test date range: 2026-01-27 00:00:00 -> 2026-04-04 00:00:00


In [14]:
def build_recency_sample_weights(df, date_col="GAME_DATE", lambda_=SAMPLE_WEIGHT_LAMBDA):
    dates = pd.to_datetime(df[date_col])
    max_date = dates.max()
    age_days = (max_date - dates).dt.days
    weights = np.exp(-lambda_ * age_days)
    return pd.Series(weights, index=df.index, name="sample_weight")

EXCLUDE_COLS = [
    "TOTAL_POINTS",
    "LINE_ERROR",
    "SEASON_YEAR",
    "GAME_DATE",
]

X_dev = df_dev.drop(columns=EXCLUDE_COLS, errors="ignore")
y_dev = pd.to_numeric(df_dev[TARGET_COL], errors="coerce")
sample_weight_dev = build_recency_sample_weights(df_dev)

X_test_final = df_test_final.drop(columns=EXCLUDE_COLS, errors="ignore")
y_test_final = pd.to_numeric(df_test_final[TARGET_COL], errors="coerce")

print(f"X_dev shape: {X_dev.shape}")
print(f"X_test_final shape: {X_test_final.shape}")
print(
    f"Recency sample weights lambda={SAMPLE_WEIGHT_LAMBDA}: "
    f"min={sample_weight_dev.min():.4f}, max={sample_weight_dev.max():.4f}"
)


X_dev shape: (5341, 2051)
X_test_final shape: (471, 2051)
Recency sample weights lambda=0.0075: min=0.0000, max=1.0000


In [15]:
ou_scorer = OverUnderScorerLineError()
ou_scorer_edge_2 = OverUnderScorerLineErrorMinEdge(min_edge=2)
ou_scorer_edge_4 = OverUnderScorerLineErrorMinEdge(min_edge=4)

scoring = {
    "MSE": "neg_mean_squared_error",
    "RMSE": "neg_root_mean_squared_error",
    "MAE": "neg_mean_absolute_error",
    "R2": "r2",
    "OU_Betting_Accuracy": ou_scorer,
    "OU_Betting_Accuracy_Edge_2": ou_scorer_edge_2,
    "OU_Betting_Accuracy_Edge_4": ou_scorer_edge_4,
}


def print_metrics(cv_results):
    for sc in scoring.keys():
        train_key = f"train_{sc}"
        test_key = f"test_{sc}"

        train_val = cv_results[train_key].mean()
        test_val = cv_results[test_key].mean()

        if sc in {"MSE", "RMSE", "MAE"}:
            train_val = -train_val
            test_val = -test_val

        if sc.startswith("OU_Betting_Accuracy"):
            print(f"Train {sc}: {train_val:.2%}")
            print(f"Validation {sc}: {test_val:.2%}")
        else:
            print(f"Train {sc}: {train_val:.5f}")
            print(f"Validation {sc}: {test_val:.5f}")
        print()


In [16]:
DAY_BY_DAY_METRIC_NAME = "OU_Betting_Accuracy"
DAY_BY_DAY_THRESHOLDS = (1, 2, 3)


def summarize_walk_forward_thresholds(predictions_df, thresholds=DAY_BY_DAY_THRESHOLDS):
    y_true = pd.to_numeric(predictions_df["y_true"], errors="coerce").to_numpy(dtype=float)
    y_pred = pd.to_numeric(predictions_df["y_pred"], errors="coerce").to_numpy(dtype=float)
    margin = np.abs(y_pred)
    n_total = len(predictions_df)

    rows = []
    for t in thresholds:
        mask = margin > t
        n = int(mask.sum())
        acc = (
            np.nan
            if n == 0
            else over_under_betting_accuracy_error_line(
                y_true_error=y_true[mask],
                y_pred_error=y_pred[mask],
            )
        )
        rows.append(
            {
                "threshold_abs_pred_error_gt": t,
                "n_games": n,
                "pct_of_test": (n / n_total) if n_total else np.nan,
                "directional_accuracy": acc,
            }
        )

    return pd.DataFrame(rows)


def run_day_by_day_walk_forward_evaluation(
    *,
    label,
    df_dev,
    df_test_final,
    fit_and_predict,
    max_games=TRAIN_GAMES,
    metric_name=DAY_BY_DAY_METRIC_NAME,
    thresholds=DAY_BY_DAY_THRESHOLDS,
):
    result = evaluate_day_by_day_walk_forward(
        df_dev=df_dev,
        df_test_final=df_test_final,
        fit_and_predict=fit_and_predict,
        metric_fn=lambda y_true, y_pred: over_under_betting_accuracy_error_line(
            y_true_error=y_true,
            y_pred_error=y_pred,
        ),
        target_col=TARGET_COL,
        max_games=max_games,
        metric_name=metric_name,
    )

    threshold_results = summarize_walk_forward_thresholds(
        result.predictions,
        thresholds=thresholds,
    )

    print(f"{label} mean day-by-day {metric_name}: {result.mean_metric:.2%}")
    display(result.daily_results.style.format({metric_name: "{:.2%}"}))
    print(f"{label} thresholded walk-forward accuracy")
    display(
        threshold_results.style.format(
            {"pct_of_test": "{:.1%}", "directional_accuracy": "{:.2%}"}
        )
    )
    return result, threshold_results


In [17]:
splits, fold_info = make_test_anchored_walk_forward_splits(
    df=df_dev,
    date_col="GAME_DATE",
    season_col="SEASON_YEAR",
    test_games=25,
    step_games_between_tests=50,
    train_games=TRAIN_GAMES,
    min_train_games=TRAIN_GAMES*0.75,
    max_folds=15,
    verbose=1,
)

assert_valid_time_splits(df_dev, splits)



Created 15 test-anchored walk-forward folds
 fold  train_n_games  test_n_games train_start_date train_end_date test_start_date test_end_date  test_season
    1           2500            28       2022-12-14     2025-01-25      2025-01-26    2025-01-29         2024
    2           2500            27       2022-12-26     2025-02-05      2025-02-06    2025-02-09         2024
    3           2500            30       2023-01-06     2025-02-21      2025-02-22    2025-02-25         2024
    4           2500            30       2023-01-17     2025-03-04      2025-03-05    2025-03-08         2024
    5           2500            33       2023-01-29     2025-03-15      2025-03-16    2025-03-19         2024
    6           2500            30       2023-02-10     2025-03-26      2025-03-27    2025-03-30         2024
    7           2500            27       2023-02-28     2025-04-06      2025-04-07    2025-04-10         2024
    8           2500            32       2023-03-18     2025-06-22      2025

In [18]:
season_bl = DummyRegressor(strategy="mean")

cv_results = cross_validate(
    season_bl,
    X_dev,
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=1,
)

print("DummyRegressor baseline")
print_metrics(cv_results)

DummyRegressor baseline
Train MSE: 283.98762
Validation MSE: 308.34897

Train RMSE: 16.85144
Validation RMSE: 17.43618

Train MAE: 13.45970
Validation MAE: 13.76841

Train R2: 0.00000
Validation R2: -0.04046

Train OU_Betting_Accuracy: 51.53%
Validation OU_Betting_Accuracy: 52.54%

Train OU_Betting_Accuracy_Edge_2: 0.00%
Validation OU_Betting_Accuracy_Edge_2: 0.00%

Train OU_Betting_Accuracy_Edge_4: 0.00%
Validation OU_Betting_Accuracy_Edge_4: 0.00%



In [19]:
lr = LinearRegression()

cv_results = cross_validate(
    lr,
    X_dev.fillna(0),
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1,
)

print("Linear Regression")
print_metrics(cv_results)

Linear Regression
Train MSE: 113.01837
Validation MSE: 839430.61996

Train RMSE: 10.62922
Validation RMSE: 273.65209

Train MAE: 8.31874
Validation MAE: 86.00730

Train R2: 0.60211
Validation R2: -1906.13185

Train OU_Betting_Accuracy: 79.13%
Validation OU_Betting_Accuracy: 48.21%

Train OU_Betting_Accuracy_Edge_2: 82.61%
Validation OU_Betting_Accuracy_Edge_2: 49.01%

Train OU_Betting_Accuracy_Edge_4: 85.78%
Validation OU_Betting_Accuracy_Edge_4: 49.31%



In [20]:
xgb_reg_no_weights = XGBRegressor(
    max_depth=3,
    learning_rate=0.05,
    n_estimators=50,
    subsample=0.65,
    colsample_bytree=0.68,
    reg_alpha=5.28,
    reg_lambda=1.3,
    min_child_weight=5.08,
    gamma=0.0085,
    n_jobs=-1,
    random_state=16,
)

cv_results_no_weights = cross_validate(
    xgb_reg_no_weights,
    X_dev,
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=1,
)

print("XGBoost no sample weights")
print_metrics(cv_results_no_weights)

XGBoost no sample weights
Train MSE: 237.26551
Validation MSE: 307.23433

Train RMSE: 15.40298
Validation RMSE: 17.40255

Train MAE: 12.30677
Validation MAE: 13.65607

Train R2: 0.16451
Validation R2: -0.03668

Train OU_Betting_Accuracy: 72.44%
Validation OU_Betting_Accuracy: 55.51%

Train OU_Betting_Accuracy_Edge_2: 88.36%
Validation OU_Betting_Accuracy_Edge_2: 60.83%

Train OU_Betting_Accuracy_Edge_4: 97.29%
Validation OU_Betting_Accuracy_Edge_4: 27.78%



In [21]:
xgb_reg_no_weights.fit(X_dev, y_dev)

y_pred_test_error = xgb_reg_no_weights.predict(X_test_final)

mse = mean_squared_error(y_test_final, y_pred_test_error)
rmse = root_mean_squared_error(y_test_final, y_pred_test_error)
mae = mean_absolute_error(y_test_final, y_pred_test_error)
ou_acc = over_under_betting_accuracy_error_line(
    y_true_error=y_test_final,
    y_pred_error=y_pred_test_error,
)
ou_acc_edge_2 = over_under_betting_accuracy_error_line_with_min_edge(
    y_true_error=y_test_final,
    y_pred_error=y_pred_test_error,
    min_edge=2,
)
ou_acc_edge_4 = over_under_betting_accuracy_error_line_with_min_edge(
    y_true_error=y_test_final,
    y_pred_error=y_pred_test_error,
    min_edge=4,
)

print("Final test metrics")
print(f"MSE: {mse:.5f}")
print(f"RMSE: {rmse:.5f}")
print(f"MAE: {mae:.5f}")
print(f"OU_Betting_Accuracy: {ou_acc:.2%}")
print(f"OU_Betting_Accuracy_Edge_2: {ou_acc_edge_2:.2%}")
print(f"OU_Betting_Accuracy_Edge_4: {ou_acc_edge_4:.2%}")

Final test metrics
MSE: 306.51899
RMSE: 17.50768
MAE: 13.78556
OU_Betting_Accuracy: 54.66%
OU_Betting_Accuracy_Edge_2: 56.60%
OU_Betting_Accuracy_Edge_4: 100.00%


In [22]:
results_df, y_pred_test_error = evaluate_error_thresholds(
    model=xgb_reg_no_weights,
    X_test=X_test_final,
    y_test_error=y_test_final,
    thresholds=range(0, 11),
)

display(
    results_df.style.format(
        {"pct_of_test": "{:.1%}", "directional_accuracy": "{:.2%}"}
    )
)


,threshold_abs_pred_error_gt,n_games,pct_of_test,directional_accuracy
0,0,471,100.0%,54.66%
1,1,204,43.3%,52.53%
2,2,57,12.1%,56.60%
3,3,9,1.9%,44.44%
4,4,1,0.2%,100.00%
5,5,1,0.2%,100.00%
6,6,0,0.0%,nan%
7,7,0,0.0%,nan%
8,8,0,0.0%,nan%
9,9,0,0.0%,nan%


In [23]:
def fit_and_predict_xgb_no_weights_day_by_day(train_df, test_df):
    model = XGBRegressor(**xgb_reg_no_weights.get_params())

    X_train = train_df.drop(columns=EXCLUDE_COLS, errors="ignore")
    y_train = pd.to_numeric(train_df[TARGET_COL], errors="coerce")
    X_test = test_df.drop(columns=EXCLUDE_COLS, errors="ignore")

    model.fit(X_train, y_train)
    return model.predict(X_test)


day_by_day_no_weights, day_by_day_no_weights_thresholds = run_day_by_day_walk_forward_evaluation(
    label="XGBoost no sample weights",
    df_dev=df_dev,
    df_test_final=df_test_final,
    fit_and_predict=fit_and_predict_xgb_no_weights_day_by_day,
    max_games=TRAIN_GAMES,
)

XGBoost no sample weights mean day-by-day OU_Betting_Accuracy: 53.69%


,date,train_n_games,test_n_games,train_start_date,train_end_date,OU_Betting_Accuracy
0,2026-01-27 00:00:00,2500,7,2024-01-12 00:00:00,2026-01-26 00:00:00,66.67%
1,2026-01-28 00:00:00,2500,9,2024-01-14 00:00:00,2026-01-27 00:00:00,66.67%
2,2026-01-29 00:00:00,2500,8,2024-01-16 00:00:00,2026-01-28 00:00:00,50.00%
3,2026-01-30 00:00:00,2500,9,2024-01-19 00:00:00,2026-01-29 00:00:00,50.00%
4,2026-01-31 00:00:00,2500,6,2024-01-27 00:00:00,2026-01-30 00:00:00,66.67%
5,2026-02-01 00:00:00,2500,10,2024-01-29 00:00:00,2026-01-31 00:00:00,60.00%
6,2026-02-02 00:00:00,2500,4,2024-01-30 00:00:00,2026-02-01 00:00:00,50.00%
7,2026-02-03 00:00:00,2500,10,2024-01-31 00:00:00,2026-02-02 00:00:00,30.00%
8,2026-02-04 00:00:00,2500,7,2024-02-01 00:00:00,2026-02-03 00:00:00,42.86%
9,2026-02-05 00:00:00,2500,8,2024-02-02 00:00:00,2026-02-04 00:00:00,62.50%


XGBoost no sample weights thresholded walk-forward accuracy


,threshold_abs_pred_error_gt,n_games,pct_of_test,directional_accuracy
0,1,279,59.2%,53.85%
1,2,122,25.9%,54.17%
2,3,49,10.4%,53.06%


## Check weighted

In [24]:
xgb_reg_weights = XGBRegressor(
    max_depth=3,
    learning_rate=0.05,
    n_estimators=50,
    subsample=0.65,
    colsample_bytree=0.68,
    reg_alpha=5.28,
    reg_lambda=1.3,
    min_child_weight=5.08,
    gamma=0.0085,
    n_jobs=-1,
    random_state=16,
)

weighted_xgb = TemporalDecaySampleWeightRegressor(
    estimator=xgb_reg_weights,
    dates=df_dev["GAME_DATE"],
    lambda_=SAMPLE_WEIGHT_LAMBDA,
)

cv_results_weights = cross_validate(
    weighted_xgb,
    X_dev,
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=1,
)

print("XGBoost with sample weights (per-fold decay)")
print_metrics(cv_results_weights)

XGBoost with sample weights (per-fold decay)
Train MSE: 252.33635
Validation MSE: 310.66005

Train RMSE: 15.88478
Validation RMSE: 17.51481

Train MAE: 12.60880
Validation MAE: 13.80935

Train R2: 0.11131
Validation R2: -0.05216

Train OU_Betting_Accuracy: 63.08%
Validation OU_Betting_Accuracy: 51.16%

Train OU_Betting_Accuracy_Edge_2: 73.33%
Validation OU_Betting_Accuracy_Edge_2: 54.37%

Train OU_Betting_Accuracy_Edge_4: 85.45%
Validation OU_Betting_Accuracy_Edge_4: 49.52%



In [25]:
weighted_xgb.fit(X_dev, y_dev)

y_pred_test_error = weighted_xgb.predict(X_test_final)

mse = mean_squared_error(y_test_final, y_pred_test_error)
rmse = root_mean_squared_error(y_test_final, y_pred_test_error)
mae = mean_absolute_error(y_test_final, y_pred_test_error)
ou_acc = over_under_betting_accuracy_error_line(
    y_true_error=y_test_final,
    y_pred_error=y_pred_test_error,
)
ou_acc_edge_2 = over_under_betting_accuracy_error_line_with_min_edge(
    y_true_error=y_test_final,
    y_pred_error=y_pred_test_error,
    min_edge=2,
)
ou_acc_edge_4 = over_under_betting_accuracy_error_line_with_min_edge(
    y_true_error=y_test_final,
    y_pred_error=y_pred_test_error,
    min_edge=4,
)

print("Final test metrics")
print(f"MSE: {mse:.5f}")
print(f"RMSE: {rmse:.5f}")
print(f"MAE: {mae:.5f}")
print(f"OU_Betting_Accuracy: {ou_acc:.2%}")
print(f"OU_Betting_Accuracy_Edge_2: {ou_acc_edge_2:.2%}")
print(f"OU_Betting_Accuracy_Edge_4: {ou_acc_edge_4:.2%}")

Final test metrics
MSE: 314.42478
RMSE: 17.73203
MAE: 13.93926
OU_Betting_Accuracy: 51.41%
OU_Betting_Accuracy_Edge_2: 51.95%
OU_Betting_Accuracy_Edge_4: 55.17%


In [26]:
results_df, y_pred_test_error = evaluate_error_thresholds(
    model=weighted_xgb,
    X_test=X_test_final,
    y_test_error=y_test_final,
    thresholds=range(0, 11),
)

display(
    results_df.style.format(
        {"pct_of_test": "{:.1%}", "directional_accuracy": "{:.2%}"}
    )
)


,threshold_abs_pred_error_gt,n_games,pct_of_test,directional_accuracy
0,0,471,100.0%,51.41%
1,1,353,74.9%,51.15%
2,2,260,55.2%,51.95%
3,3,165,35.0%,51.85%
4,4,90,19.1%,55.17%
5,5,41,8.7%,55.00%
6,6,15,3.2%,50.00%
7,7,3,0.6%,66.67%
8,8,3,0.6%,66.67%
9,9,1,0.2%,0.00%


In [27]:
def fit_and_predict_xgb_weights_day_by_day(train_df, test_df):
    base_model = XGBRegressor(**xgb_reg_weights.get_params())
    model = TemporalDecaySampleWeightRegressor(
        estimator=base_model,
        dates=train_df["GAME_DATE"],
        lambda_=SAMPLE_WEIGHT_LAMBDA,
    )

    X_train = train_df.drop(columns=EXCLUDE_COLS, errors="ignore")
    y_train = pd.to_numeric(train_df[TARGET_COL], errors="coerce")
    X_test = test_df.drop(columns=EXCLUDE_COLS, errors="ignore")

    model.fit(X_train, y_train)
    return model.predict(X_test)


day_by_day_weights, day_by_day_weights_thresholds = run_day_by_day_walk_forward_evaluation(
    label="XGBoost with sample weights",
    df_dev=df_dev,
    df_test_final=df_test_final,
    fit_and_predict=fit_and_predict_xgb_weights_day_by_day,
    max_games=TRAIN_GAMES,
)


XGBoost with sample weights mean day-by-day OU_Betting_Accuracy: 54.14%


,date,train_n_games,test_n_games,train_start_date,train_end_date,OU_Betting_Accuracy
0,2026-01-27 00:00:00,2500,7,2024-01-12 00:00:00,2026-01-26 00:00:00,83.33%
1,2026-01-28 00:00:00,2500,9,2024-01-14 00:00:00,2026-01-27 00:00:00,55.56%
2,2026-01-29 00:00:00,2500,8,2024-01-16 00:00:00,2026-01-28 00:00:00,62.50%
3,2026-01-30 00:00:00,2500,9,2024-01-19 00:00:00,2026-01-29 00:00:00,50.00%
4,2026-01-31 00:00:00,2500,6,2024-01-27 00:00:00,2026-01-30 00:00:00,50.00%
5,2026-02-01 00:00:00,2500,10,2024-01-29 00:00:00,2026-01-31 00:00:00,70.00%
6,2026-02-02 00:00:00,2500,4,2024-01-30 00:00:00,2026-02-01 00:00:00,75.00%
7,2026-02-03 00:00:00,2500,10,2024-01-31 00:00:00,2026-02-02 00:00:00,40.00%
8,2026-02-04 00:00:00,2500,7,2024-02-01 00:00:00,2026-02-03 00:00:00,57.14%
9,2026-02-05 00:00:00,2500,8,2024-02-02 00:00:00,2026-02-04 00:00:00,75.00%


XGBoost with sample weights thresholded walk-forward accuracy


,threshold_abs_pred_error_gt,n_games,pct_of_test,directional_accuracy
0,1,349,74.1%,56.43%
1,2,244,51.8%,55.19%
2,3,158,33.5%,55.41%


# Optuna

In [29]:
study = tune_xgb_error_line_optuna(
    X=X_dev,
    y=y_dev,
    sample_weight_dates=df_dev["GAME_DATE"],
    tune_sample_weight_lambda=True,
    sample_weight_lambda_bounds=SAMPLE_WEIGHT_LAMBDA_BOUNDS,
    splits=splits,
    n_trials=30,
    timeout=1.5 * 3600,
    # timeout=600,

    objective_name="reg:squarederror",
    study_name="xgb_error_line_mae",
)

best_trial_lexi = select_best_trial_lexicographic(
    study,
    mae_tolerance_abs=0.05,
)

print("Optuna best by MAE only")
print("Trial:", study.best_trial.number)
print("Best CV MAE:", study.best_value)
print("Mean OU accuracy:", study.best_trial.user_attrs.get("mean_ou_acc"))
print("Mean OU accuracy edge 2:", study.best_trial.user_attrs.get("mean_ou_acc_edge_2"))
print("Mean OU accuracy edge 4:", study.best_trial.user_attrs.get("mean_ou_acc_edge_4"))

print("\nSelected trial after MAE-first / OU-second ranking")
print("Trial:", best_trial_lexi.number)
print("CV MAE:", best_trial_lexi.user_attrs.get("mean_mae", best_trial_lexi.value))
print("Mean RMSE:", best_trial_lexi.user_attrs.get("mean_rmse"))
print("Mean R2:", best_trial_lexi.user_attrs.get("mean_r2"))
print("Mean OU accuracy:", best_trial_lexi.user_attrs.get("mean_ou_acc"))
print("Mean OU accuracy edge 2:", best_trial_lexi.user_attrs.get("mean_ou_acc_edge_2"))
print("Mean OU accuracy edge 4:", best_trial_lexi.user_attrs.get("mean_ou_acc_edge_4"))
print("Median best_iteration:", best_trial_lexi.user_attrs.get("median_best_iteration"))
print("Params:")
for k, v in best_trial_lexi.params.items():
    print(f"{k}: {v}")

trials_df = summarize_optuna_trials(study)
display(
    trials_df.head(15).style.format(
        {
            "value_mae": "{:.4f}",
            "mean_rmse": "{:.4f}",
            "mean_r2": "{:.4f}",
            "mean_ou_acc": "{:.2%}",
            "mean_ou_acc_edge_2": "{:.2%}",
            "mean_ou_acc_edge_4": "{:.2%}",
        }
    )
)

candidates_df = summarize_lexicographic_candidates(
    study,
    mae_tolerance_abs=0.05,
)
display(
    candidates_df.head(15).style.format(
        {
            "value_mae": "{:.4f}",
            "mean_mae": "{:.4f}",
            "mean_rmse": "{:.4f}",
            "mean_r2": "{:.4f}",
            "mean_ou_acc": "{:.2%}",
            "mean_ou_acc_edge_2": "{:.2%}",
            "mean_ou_acc_edge_4": "{:.2%}",
        }
    )
)


[I 2026-04-06 13:00:50,650] A new study created in memory with name: xgb_error_line_mae


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-04-06 13:08:40,290] Trial 0 finished with value: 13.589075659486722 and parameters: {'max_depth': 2, 'min_child_weight': 18.346704707583235, 'gamma': 1.6970342240854253, 'subsample': 0.5682407800531226, 'colsample_bytree': 0.5123279759064977, 'learning_rate': 0.011926786034588454, 'reg_alpha': 1.8771791376898666, 'reg_lambda': 1.897469395521307, 'sample_weight_lambda': 0.0001422437941448231}. Best is trial 0 with value: 13.589075659486722.
[I 2026-04-06 13:14:08,045] Trial 1 finished with value: 13.560314044667074 and parameters: {'max_depth': 4, 'min_child_weight': 20.290108931287893, 'gamma': 0.3261777842309625, 'subsample': 0.8390562044499359, 'colsample_bytree': 0.4213034780854249, 'learning_rate': 0.012620826760486504, 'reg_alpha': 0.09307011182812809, 'reg_lambda': 15.258811505244246, 'sample_weight_lambda': 0.0010239553604139541}. Best is trial 1 with value: 13.560314044667074.
[I 2026-04-06 13:17:08,773] Trial 2 finished with value: 13.565693460478636 and parameters: {'

,trial,value_mae,mean_rmse,mean_r2,mean_ou_acc,mean_ou_acc_edge_2,mean_ou_acc_edge_3,mean_ou_acc_edge_4,mean_best_iteration,median_best_iteration,max_depth,min_child_weight,gamma,subsample,colsample_bytree,learning_rate,reg_alpha,reg_lambda,sample_weight_lambda
0,13,13.3175,17.1791,-0.0121,58.55%,55.89%,0.513752,32.23%,63,11,4,26.302262,1.442185,0.857761,0.792575,0.059448,0.345644,3.750732,0.000127
1,6,13.3438,17.2099,-0.0148,57.48%,57.62%,0.336296,35.50%,104,30,4,21.354130,1.318300,0.833701,0.787420,0.040420,0.264001,42.379142,0.000127
2,7,13.3519,17.1905,-0.0129,61.46%,44.26%,0.402647,28.00%,116,47,4,26.800109,1.628315,0.776795,0.675325,0.031011,4.220375,1.554967,0.000271
3,14,13.3887,17.1822,-0.0103,57.92%,48.84%,0.272619,27.22%,55,9,3,46.459313,1.349062,0.870136,0.775350,0.059010,0.449913,3.759260,0.000112
4,16,13.4255,17.0577,0.0048,58.08%,45.93%,0.484791,41.28%,39,11,4,23.157486,2.084847,0.704882,0.734214,0.054937,0.051579,5.393719,0.014981
5,11,13.4307,17.2711,-0.0223,57.86%,37.40%,0.426781,22.67%,92,26,4,37.458731,1.863428,0.817635,0.794623,0.030033,0.225497,3.351751,0.000273
6,4,13.4554,17.2189,-0.0144,60.31%,38.76%,0.313588,26.81%,68,22,3,27.769564,0.863656,0.554592,0.697083,0.040937,0.028904,9.594362,0.000551
7,15,13.4672,17.2267,-0.0169,57.45%,41.04%,0.254815,28.22%,82,31,3,5.560352,0.708417,0.880063,0.535979,0.043531,0.145565,30.307153,0.002462
8,12,13.4819,17.2856,-0.0235,56.47%,38.14%,0.323911,26.44%,106,47,4,8.380570,2.158791,0.765014,0.661528,0.029101,14.588800,47.613477,0.000257
9,10,13.4990,17.2416,-0.0181,57.34%,38.69%,0.288434,36.48%,106,58,3,56.486523,2.911610,0.748320,0.753588,0.031302,19.122169,44.555302,0.000120


,trial,value_mae,mean_mae,mean_rmse,mean_r2,mean_ou_acc,mean_ou_acc_edge_2,mean_ou_acc_edge_3,mean_ou_acc_edge_4,mean_best_iteration,median_best_iteration,mae_cutoff,max_depth,min_child_weight,gamma,subsample,colsample_bytree,learning_rate,reg_alpha,reg_lambda,sample_weight_lambda
0,7,13.3519,13.3519,17.1905,-0.0129,61.46%,44.26%,0.402647,28.00%,116,47,13.367545,4,26.800109,1.628315,0.776795,0.675325,0.031011,4.220375,1.554967,0.000271
1,13,13.3175,13.3175,17.1791,-0.0121,58.55%,55.89%,0.513752,32.23%,63,11,13.367545,4,26.302262,1.442185,0.857761,0.792575,0.059448,0.345644,3.750732,0.000127
2,6,13.3438,13.3438,17.2099,-0.0148,57.48%,57.62%,0.336296,35.50%,104,30,13.367545,4,21.354130,1.318300,0.833701,0.787420,0.040420,0.264001,42.379142,0.000127


In [30]:
def fit_and_predict_optuna_day_by_day(train_df, test_df):
    X_train = train_df.drop(columns=EXCLUDE_COLS, errors="ignore")
    y_train = pd.to_numeric(train_df[TARGET_COL], errors="coerce")
    X_test = test_df.drop(columns=EXCLUDE_COLS, errors="ignore")

    model = fit_best_xgb_error_line(
        X_dev=X_train,
        y_dev=y_train,
        sample_weight_dates=train_df["GAME_DATE"],
        sample_weight_lambda=best_trial_lexi.params.get("sample_weight_lambda"),
        trial=best_trial_lexi,
        objective_name="reg:squarederror",
    )
    return model.predict(X_test)

day_by_day_optuna, day_by_day_optuna_thresholds = run_day_by_day_walk_forward_evaluation(
    label="Optuna-selected XGBoost",
    df_dev=df_dev,
    df_test_final=df_test_final,
    fit_and_predict=fit_and_predict_optuna_day_by_day,
    max_games=TRAIN_GAMES,
)

Optuna-selected XGBoost mean day-by-day OU_Betting_Accuracy: 56.26%


,date,train_n_games,test_n_games,train_start_date,train_end_date,OU_Betting_Accuracy
0,2026-01-27 00:00:00,2500,7,2024-01-12 00:00:00,2026-01-26 00:00:00,66.67%
1,2026-01-28 00:00:00,2500,9,2024-01-14 00:00:00,2026-01-27 00:00:00,77.78%
2,2026-01-29 00:00:00,2500,8,2024-01-16 00:00:00,2026-01-28 00:00:00,87.50%
3,2026-01-30 00:00:00,2500,9,2024-01-19 00:00:00,2026-01-29 00:00:00,50.00%
4,2026-01-31 00:00:00,2500,6,2024-01-27 00:00:00,2026-01-30 00:00:00,33.33%
5,2026-02-01 00:00:00,2500,10,2024-01-29 00:00:00,2026-01-31 00:00:00,60.00%
6,2026-02-02 00:00:00,2500,4,2024-01-30 00:00:00,2026-02-01 00:00:00,25.00%
7,2026-02-03 00:00:00,2500,10,2024-01-31 00:00:00,2026-02-02 00:00:00,30.00%
8,2026-02-04 00:00:00,2500,7,2024-02-01 00:00:00,2026-02-03 00:00:00,71.43%
9,2026-02-05 00:00:00,2500,8,2024-02-02 00:00:00,2026-02-04 00:00:00,37.50%


Optuna-selected XGBoost thresholded walk-forward accuracy


,threshold_abs_pred_error_gt,n_games,pct_of_test,directional_accuracy
0,1,251,53.3%,57.72%
1,2,107,22.7%,58.49%
2,3,27,5.7%,55.56%


In [31]:
total_df = df_dev.tail(TRAIN_GAMES)

In [32]:
X_dev = total_df.drop(columns=EXCLUDE_COLS, errors="ignore")
y_dev = pd.to_numeric(total_df[TARGET_COL], errors="coerce")
sample_weight_dates_dev = total_df["GAME_DATE"]


In [33]:
best_model = fit_best_xgb_error_line(
    X_dev=X_dev,
    y_dev=y_dev,
    sample_weight_dates=sample_weight_dates_dev,
    sample_weight_lambda=best_trial_lexi.params.get("sample_weight_lambda"),
    trial=best_trial_lexi,
    objective_name="reg:squarederror",
)

y_pred_test_error = best_model.predict(X_test_final)

mse = mean_squared_error(y_test_final, y_pred_test_error)
rmse = root_mean_squared_error(y_test_final, y_pred_test_error)
mae = mean_absolute_error(y_test_final, y_pred_test_error)
ou_acc = over_under_betting_accuracy_error_line(
    y_true_error=y_test_final,
    y_pred_error=y_pred_test_error,
)
ou_acc_edge_2 = over_under_betting_accuracy_error_line_with_min_edge(
    y_true_error=y_test_final,
    y_pred_error=y_pred_test_error,
    min_edge=2,
)
ou_acc_edge_4 = over_under_betting_accuracy_error_line_with_min_edge(
    y_true_error=y_test_final,
    y_pred_error=y_pred_test_error,
    min_edge=4,
)

print("Final test metrics")
print(f"MSE: {mse:.5f}")
print(f"RMSE: {rmse:.5f}")
print(f"MAE: {mae:.5f}")
print(f"OU_Betting_Accuracy: {ou_acc:.2%}")
print(f"OU_Betting_Accuracy_Edge_2: {ou_acc_edge_2:.2%}")
print(f"OU_Betting_Accuracy_Edge_4: {ou_acc_edge_4:.2%}")

Final test metrics
MSE: 308.40593
RMSE: 17.56149
MAE: 13.76880
OU_Betting_Accuracy: 55.10%
OU_Betting_Accuracy_Edge_2: 52.58%
OU_Betting_Accuracy_Edge_4: 62.50%


In [ ]:
from nba_ou.modeling.modeling import ModelBundleMetadata, ModelInfo, TrainingMetrics

df_to_train_split_rows = df_to_train.copy()
df_to_train_split_rows = df_to_train_split_rows.tail(TRAIN_GAMES)

X_full = df_to_train_split_rows.drop(columns=EXCLUDE_COLS, errors="ignore")
y_full = pd.to_numeric(df_to_train_split_rows[TARGET_COL], errors="coerce")
sample_weight_dates_full = df_to_train_split_rows["GAME_DATE"]

production_model = fit_best_xgb_error_line(
    X_dev=X_full,
    y_dev=y_full,
    sample_weight_dates=sample_weight_dates_full,
    sample_weight_lambda=best_trial_lexi.params.get("sample_weight_lambda"),
    trial=best_trial_lexi,
    objective_name="reg:squarederror",
)

latest_training_date = pd.to_datetime(df_to_train_split_rows["GAME_DATE"]).max()
model_version = latest_training_date.strftime("%d_%m_%y")
model_name = f"three_seasons_xgb_line_error_{model_version}"

metadata = ModelBundleMetadata(
    model_info=ModelInfo(
        name=model_name,
        model_version=model_version,
        model_type="three_seasons_line_error",
        prediction_source="three_seasons_xgb_line_error",
        training_code_tag="1.0",
    ),
    training_metrics=TrainingMetrics(
        best_params=best_trial_lexi.params,
        selected_trial_number=best_trial_lexi.number,
        mean_best_iteration=best_trial_lexi.user_attrs.get("mean_best_iteration"),
        median_best_iteration=best_trial_lexi.user_attrs.get("median_best_iteration"),
        cv_mae=float(best_trial_lexi.user_attrs.get("mean_mae", best_trial_lexi.value)),
        cv_rmse=best_trial_lexi.user_attrs.get("mean_rmse"),
        cv_ou_acc=best_trial_lexi.user_attrs.get("mean_ou_acc"),
        final_test_mae=float(mae),
        final_test_rmse=float(rmse),
        final_test_ou_acc=float(ou_acc),
        nan_threshold=nan_threshold,
        max_na_per_row=max_na_per_row,
        train_date_min=df_to_train_split_rows["GAME_DATE"].min().to_pydatetime(),
        train_date_max=df_to_train_split_rows["GAME_DATE"].max().to_pydatetime(),
        train_games= TRAIN_GAMES,
        sample_weight_lambda_bounds=SAMPLE_WEIGHT_LAMBDA_BOUNDS,
    ),
)

model_path, meta_path = save_model_bundle(
    model=production_model,
    feature_names=list(X_full.columns),
    out_dir="/home/adrian_alvarez/Projects/NBA_over_under_predictor/models/line_error/3_seasons/",
    metadata=metadata,
)

print(
    f"Production model trained on {len(X_full)} rows using fixed n_estimators from median_best_iteration."
)
print("Saved model :", model_path)
print("Saved metadata:", meta_path)

Production model trained on 2500 rows using fixed n_estimators from median_best_iteration.
Saved model : /home/adrian_alvarez/Projects/NBA_over_under_predictor/models/line_error/3_seasons/three_seasons_xgb_line_error_04_04_26.json
Saved metadata: /home/adrian_alvarez/Projects/NBA_over_under_predictor/models/line_error/3_seasons/three_seasons_xgb_line_error_04_04_26.meta.json


In [35]:
best_trial_lexi.user_attrs.get("median_best_iteration")


47